# Simulation Dataset Preparation

This notebook was used to create ACCIDENT synthetic dataset.

## Overview
- Load raw data
- Clean and preprocess
- Generate simulation features
- Export processed dataset



In [ ]:
import sys
from collections import defaultdict
sys.path.append("../")

import os
import os.path as osp
import subprocess
import cv2
import numpy as np
import pandas as pd
import shutil
import glob

from pathlib import Path
from tqdm import tqdm
from src.client.core.ioutils import load_yaml, load_json, save_json
from joblib import Parallel, delayed

In [ ]:
def read_file(file_name: str) -> list[str]:
    with open(file_name, "r") as f:
        return f.readlines()

def is_valid_path(s: str) -> bool:
    try:
        Path(s)
        return True
    except (TypeError, ValueError):
        return False

### Filter manually selected wrong simulations

In [ ]:
POSSIBLE_PREFIXES = ["rgb", "display"]

TO_REMOVE_FILE = "../runs/multiviews/to_remove.md"
lines = read_file(TO_REMOVE_FILE)

total_scenarios_removed = 0
for line in tqdm(lines, total=len(lines)):
    removed_something = False
    line = line.strip()
    if not is_valid_path(line) or "/" not in line:
        continue

    path = Path(line)
    dir_name, file_name = path.parent, path.name

    if path.exists():
        print(f"Removing original file: {path}")
        os.remove(path)
        removed_something = True
    for prefix in POSSIBLE_PREFIXES:
        if prefix in str(path):
            continue
        secondary_file = osp.join(dir_name, "_".join([prefix, *file_name.split("_")[1:]]))
        if osp.isfile(secondary_file):
            print(f"Removing secondary ({prefix}) file: {secondary_file}")
            os.remove(secondary_file)
            removed_something = True

    parts = dir_name.parts
    idx = parts.index("images")
    before_images_path = Path(*parts[:idx])
    ann_file = osp.join(before_images_path, "labels", "train", f'{path.stem.split("_")[-1]}.json')

    if osp.isfile(ann_file):
        print(f"Removing annotation file: {ann_file}")
        os.remove(ann_file)
        removed_something = True

    if removed_something:
        total_scenarios_removed += 1


print(f"{total_scenarios_removed} scenarios removed")

# Prepare dataset metadata

In [ ]:
DATASET_DIR = "../runs/multiviews"

missing = defaultdict(dict)
rows = []

scenario_names = os.listdir(DATASET_DIR)
scenario_names = [sc_name for sc_name in scenario_names if osp.isdir(osp.join(DATASET_DIR, sc_name))]
for scenario_name in tqdm(scenario_names, total=len(scenario_names)):
    video_dir = osp.join(DATASET_DIR, scenario_name)

    exp_names = os.listdir(video_dir)
    for exp_name in exp_names:
        exp_dir = osp.join(video_dir, exp_name)
        if not osp.isdir(exp_dir):
            print(f"{exp_name} does not exist")
            continue

        video_image_dir = osp.join(exp_dir, "images", "train")
        video_annotation_dir = osp.join(exp_dir, "labels", "train")

        config = load_yaml(osp.join(exp_dir, f"{osp.basename(video_dir).replace('_video', '')}.yaml"))

        unique_timestamps = {}
        video_files = os.listdir(video_image_dir)
        for video_file in video_files:
            video_type, timestamp = Path(video_file).stem.split("_")
            if timestamp not in unique_timestamps:
                unique_timestamps[timestamp] = {video_type: osp.join(video_image_dir, video_file)}
            else:
                unique_timestamps[timestamp][video_type] = osp.join(video_image_dir, video_file)

            if not osp.exists(osp.join(video_image_dir, f"rgb_{timestamp}.mp4")):
                print(osp.join(video_image_dir, f"rgb_{timestamp}.mp4"))
            if not osp.exists(osp.join(video_image_dir, f"display_{timestamp}.mp4")):
                print(osp.join(video_image_dir, f"display_{timestamp}.mp4"))

        timestamps_missing_anns = []
        for unique_timestamp in unique_timestamps.keys():
            anns_path = osp.join(video_annotation_dir, f"{unique_timestamp}.json")
            if not osp.exists(anns_path):
                timestamps_missing_anns.append(unique_timestamp)
            else:
                unique_timestamps[unique_timestamp]["annotations"] = anns_path

        missing[scenario_name][exp_name] = timestamps_missing_anns

        for unique_timestamp, _data in unique_timestamps.items():
            rows.append({
                "scenario": scenario_name,
                "experiment": exp_name,
                "timestamp": unique_timestamp,
                "rgb_path": _data.get("rgb", None),
                "display_path": _data.get("display", None),
                "annotations_path": _data.get("annotations", None),
            })

df = pd.DataFrame(rows)
# missing

## Filter missing files

In [ ]:
df = df.drop_duplicates("timestamp")
df = df[~df["annotations_path"].isna()]
df = df[~df["display_path"].isna()]
df.reset_index(drop=True, inplace=True)
df

## Add accident info from annotations

In [ ]:
DISPLAY_SIZE = (1920, 1080)
FPS = 20
ITERATION_OFFSET = 20


def get_accident_frame_from_iteration(iteration: int) -> int:
    return iteration - ITERATION_OFFSET


def get_video_length(path: str) -> tuple[float, int, float]:
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {path}")
    fps = cap.get((cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    cap.release()
    return fps, frame_count, duration


def bbox_to_center(bbox: list[tuple[int, int]], normalize: bool = True) -> tuple[float, float]:
    center_x = bbox[0][0] + (bbox[1][0] - bbox[0][0]) / 2
    center_y = bbox[0][1] + (bbox[1][1] - bbox[0][1]) / 2
    if normalize:
        center_x /= DISPLAY_SIZE[0]
        center_y /= DISPLAY_SIZE[1]
    return center_x, center_y


def get_accident_type(scenario_name: str) -> str:
    scenario_name = scenario_name.lower()
    if "headon" in scenario_name:
        return "head-on"
    elif "single" in scenario_name:
        return "single"
    elif "tbone" in scenario_name:
        return "t-bone"
    elif "sideswipe" in scenario_name:
        return "sideswipe"
    elif "rearend" in scenario_name:
        return "rear-end"
    else:
        raise RuntimeError(f"Unknown scenario: {scenario_name}")


new_rows = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    video_path = row["rgb_path"]
    annotation_file_data = load_json(row["annotations_path"])
    collision_data = annotation_file_data["collision"]

    iteration, accident_bbox = collision_data[0]["iteration"], collision_data[0]["collision_bbox"]
    accident_frame = get_accident_frame_from_iteration(iteration)
    fps, frame_count, duration = get_video_length(video_path)
    assert round(fps) == FPS

    center_x, center_y = bbox_to_center(accident_bbox)

    type = get_accident_type(row["scenario"])

    new_rows.append({
        "type": type,
        "accident_time": round(accident_frame / FPS, 4),
        "accident_frame": accident_frame,
        "center_x": center_x,
        "center_y": center_y,
        "x1": accident_bbox[0][0] / DISPLAY_SIZE[0],
        "y1": accident_bbox[0][1] / DISPLAY_SIZE[1],
        "x2": accident_bbox[1][0] / DISPLAY_SIZE[0],
        "y2": accident_bbox[1][1] / DISPLAY_SIZE[1],
        "no_frames": frame_count,
        "duration": duration,
        "height": DISPLAY_SIZE[1],
        "width": DISPLAY_SIZE[0],
    })

new_df = pd.DataFrame(new_rows, index=df.index)
df = pd.concat([df, new_df], axis=1)
df


### Connect camera to position and weather
The precise location and weather conditions were not saved in the output simulation metadata, so it has to be inferred from the order of the camera positions.

In [ ]:
def angle_diff(a, b):
    d = a - b
    return (d + 180) % 360 - 180


def sensor_distance(obs, base):
    loc = base["location"]
    rot = base["rotation"]
    loc_s = base["location_scaling"]
    rot_s = base["rotation_scaling"]

    d_loc = np.array([
        (obs["location"]["x"] - loc["x"]) / loc_s["x"],
        (obs["location"]["y"] - loc["y"]) / loc_s["y"],
        (obs["location"]["z"] - loc["z"]) / loc_s["z"],
    ])

    # d_rot = np.array([
    #     (obs["rotation"]["pitch"] - rot["pitch"]) / rot_s["pitch"],
    #     angle_diff(obs["rotation"]["yaw"], rot["yaw"]) / rot_s["yaw"],
    #     (obs["rotation"]["roll"] - rot["roll"]) / rot_s["roll"],
    # ])

    # return np.linalg.norm(np.concatenate([d_loc, d_rot]))
    return np.linalg.norm(d_loc)


def find_sensor_index(obs_sensor, sensor_bases):
    distances = [
        sensor_distance(obs_sensor, base)
        for base in sensor_bases
    ]
    min_dist = min(distances)
    if min_dist > 3:
        print(f"Uncertain mapping: {min_dist}")
    return int(np.argmin(distances))


unique_scenarios = df["scenario"].unique()

df["map"] = None
df["location"] = None
df["weather"] = None
df["camera_index"] = None

# scenario_to_meta_mapping = {}
for scenario in unique_scenarios:
    if scenario == "Town05_L02-RearEnd_video":
        continue
    scenario_df = df[df["scenario"] == scenario]
    scenario_df = scenario_df.sort_values("timestamp")
    scenario_file_root = scenario_df["rgb_path"].iloc[0].split("/")[:5]
    scenario_meta_path = osp.join(*scenario_file_root, f"{scenario.removesuffix('_video')}.yaml" )
    print(scenario_meta_path)
    scenario_metadata = load_yaml(scenario_meta_path)

    sensor_positions = scenario_metadata["sensors"]
    weathers = scenario_metadata["weathers"]
    prev_sensor_index = None
    weather_index = 0
    for idx, row in tqdm(scenario_df.iterrows(), total=len(scenario_df)):
        timestamp = row["timestamp"]
        annotation_file_data = load_json(row["annotations_path"])

        obs_camera = annotation_file_data["sensor"]
        sensor_index = find_sensor_index(obs_camera, sensor_positions)

        if prev_sensor_index is not None:
            # sensor cycle restarted → new weather
            if sensor_index < prev_sensor_index:
                weather_index += 1

        prev_sensor_index = sensor_index

        df.loc[idx, "map"] = scenario.split("_")[0]
        df.loc[idx, "location"] = scenario.split("_")[1][:3]
        df.loc[idx, "weather"] = weathers[weather_index]
        df.loc[idx, "camera_index"] = f"{sensor_index:02d}"


### Fix Town05_L02-RearEnd_video

In [ ]:
scenario = "Town05_L02-RearEnd_video"
timestamp_threshold = "20251"
scenario_df = df[df["scenario"] == scenario]
scenario_df = scenario_df.sort_values("timestamp")
scenario_file_root = scenario_df["rgb_path"].iloc[0].split("/")[:5]

scenario_meta_path = osp.join(*scenario_file_root, f"{scenario.removesuffix('_video')}.yaml" )
scenario_metadata = load_yaml(scenario_meta_path)

scenario_meta_path_old = osp.join(*scenario_file_root, f"{scenario.removesuffix('_video')}-old.yaml" )
scenario_metadata_old = load_yaml(scenario_meta_path_old)

sensor_positions = scenario_metadata["sensors"]
weathers = scenario_metadata["weathers"]
prev_sensor_index = None
weather_index = 0

sensor_positions_old = scenario_metadata_old["sensors"]
weathers_old = scenario_metadata_old["weathers"]
prev_sensor_index_old = None
weather_index_old = 0

for idx, row in tqdm(scenario_df.iterrows(), total=len(scenario_df)):
    timestamp = row["timestamp"]
    annotation_file_data = load_json(row["annotations_path"])

    obs_camera = annotation_file_data["sensor"]

    _weather_index, _sensor_index = None, None
    if f"{timestamp}".startswith(timestamp_threshold):  # -> older
        sensor_index = find_sensor_index(obs_camera, sensor_positions_old)
        if prev_sensor_index_old is not None:
            # sensor cycle restarted → new weather
            if sensor_index < prev_sensor_index_old:
                weather_index_old += 1
        prev_sensor_index_old = sensor_index
        _weather_index, _sensor_index = weather_index_old, "1" + f"{prev_sensor_index_old:02d}"
    else:
        sensor_index = find_sensor_index(obs_camera, sensor_positions)

        if prev_sensor_index is not None:
            # sensor cycle restarted → new weather
            if sensor_index < prev_sensor_index:
                weather_index += 1

        prev_sensor_index = sensor_index
        _weather_index, _sensor_index = weather_index, "2" + f"{prev_sensor_index:02d}"

    df.loc[idx, "map"] = scenario.split("_")[0]
    df.loc[idx, "location"] = scenario.split("_")[1][:3]
    df.loc[idx, "weather"] = weathers[_weather_index]
    df.loc[idx, "camera_index"] = _sensor_index



In [ ]:
df.head()

In [ ]:
df.to_csv("./sim-dataset/multiviews.csv")

df["accident_time"].hist(bins=100, figsize=(20, 10))

# Crop simulated videos

Randomly crop videos to alter the distribution of accident times to be more natural.

In [ ]:
df = pd.read_csv("./sim-dataset/multiviews.csv")

In [ ]:
def crop_video_ffmpeg(input_path: str, output_path: str, fps: float, start_frame: int, end_frame: int, reencode: bool = False) -> None:
    """
    Crop a video between two frame numbers using ffmpeg.

    """
    start_time = start_frame / fps
    duration = (end_frame - start_frame) / fps

    if reencode:
        cmd = [
            "ffmpeg", "-y",
            "-i", input_path,
            "-ss", str(start_time),
            "-t", str(duration),
            "-an",                 # disable audio
            "-c:v", "libx264",
            output_path,
        ]
        # "-vf", f"trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS,fps={FPS}",
    else:
        cmd = [
            "ffmpeg",
            "-y",
            "-ss", str(start_time),
            "-t", str(duration),
            "-i", input_path,
            "-c", "copy",
            output_path,
        ]

    # result = subprocess.run(cmd, capture_output=True, text=True)
    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,   # bytes
    )
    # print("STDOUT:\n", result.stdout)
    # print("STDERR:\n", result.stderr)
    result.check_returncode()

    print(f"Cropped video saved to {output_path}")


In [ ]:
def get_random_start_frame_gauss(start_frame: int, collision_frame: int, sigma_ratio: float = 0.25, min_frame_diff: int = 10) -> int:
    mean = (start_frame + collision_frame) / 2
    sigma = (collision_frame - start_frame) * sigma_ratio
    frame = int(np.random.normal(mean, sigma))
    return int(np.clip(frame, a_min=start_frame, a_max=collision_frame - min_frame_diff))


def get_random_end_frame_gauss(end_frame: int, collision_frame: int, sigma_ratio: float = 0.25, min_frame_diff: int = 10) -> int:
    mean = (collision_frame + end_frame) / 2
    sigma = (end_frame - collision_frame ) * sigma_ratio
    frame = int(np.random.normal(mean, sigma))
    return int(np.clip(frame, a_min=collision_frame + min_frame_diff, a_max=end_frame))


def get_random_start_frame_beta(
    start_frame: int,
    collision_frame: int,
    alpha: float = 2.0,   # smaller = more left bias
    beta: float = 5.0,
    min_frame_diff: int = 10,
) -> int:
    span = collision_frame - start_frame - min_frame_diff
    r = np.random.beta(alpha, beta)  # left-biased
    value =  start_frame + int(r * span)
    return int(np.clip(value, a_min=start_frame, a_max=collision_frame - min_frame_diff))


def get_random_end_frame_beta(
    end_frame: int,
    collision_frame: int,
    alpha: float = 5.0,
    beta: float = 2.0,    # smaller = more right bias
    min_frame_diff: int = 10,
) -> int:
    span = end_frame - collision_frame - min_frame_diff
    r = np.random.beta(alpha, beta)  # right-biased
    frame = collision_frame + min_frame_diff + int(r * span)
    return int(np.clip(frame, a_min=collision_frame + min_frame_diff, a_max=end_frame))


In [ ]:
get_random_start_frame_beta(start_frame=0, collision_frame=30), get_random_end_frame_beta(100, 30)

In [ ]:
INPUT_DIR = "../runs/multiviews"
OUTPUT_DIR = "../runs/multiviews_cropped"

In [ ]:
def process_row(row):
    scenario_name, exp_name = row["scenario"], row["experiment"]
    rgb_path, display_path, label_path = row["rgb_path"], row["display_path"], row["annotations_path"]

    cropped_image_dir = osp.join(OUTPUT_DIR, scenario_name, exp_name, "images", "train")
    cropped_annotation_dir = osp.join(OUTPUT_DIR, scenario_name, exp_name, "labels", "train")
    os.makedirs(cropped_image_dir, exist_ok=True)
    os.makedirs(cropped_annotation_dir, exist_ok=True)

    metadata = load_json(label_path)
    base, collision, sensor  = metadata["base"], metadata["collision"], metadata["sensor"]
    first_iteration, last_iteration = base[0]["iteration"], base[-1]["iteration"]
    collision_start_iteration = int(collision[0]["iteration"])

    cropped_annotation_path = osp.join(cropped_annotation_dir, osp.basename(label_path))
    if osp.exists(cropped_annotation_path):
        _metadata = load_json(cropped_annotation_path)
        return {
            "start_iteration": _metadata["base"][0]["iteration"],
            "end_iteration": _metadata["base"][-1]["iteration"],
        }

    start_iteration = get_random_start_frame_beta(first_iteration, collision_start_iteration)
    assert start_iteration >= first_iteration
    end_iteration = get_random_end_frame_beta(last_iteration, collision_start_iteration)
    assert end_iteration <= last_iteration

    base = [frame_data for frame_data in base if start_iteration <= frame_data["iteration"] <= end_iteration]
    collision = [frame_data for frame_data in collision if frame_data["iteration"] <= end_iteration]

    start_frame = start_iteration - first_iteration
    end_frame = end_iteration - first_iteration + 1  # end iteration inclusive

    cropped_metadata = {
        "base": base,
        "collision": collision,
        "sensor": sensor,
    }
    save_json(path=osp.join(cropped_annotation_dir, osp.basename(label_path)), content=cropped_metadata)

    # videos
    for video_path in [rgb_path, display_path]:
        cropped_video_path = osp.join(cropped_image_dir, osp.basename(video_path))
        crop_video_ffmpeg(
            input_path=video_path,
            output_path=cropped_video_path,
            fps=FPS,
            start_frame=start_frame,
            end_frame=end_frame,
            reencode=True
        )

    for config in glob.glob(osp.join(INPUT_DIR, scenario_name, exp_name, "*.yaml")):
        shutil.copy(
            config,
            osp.join(OUTPUT_DIR, scenario_name, exp_name, osp.basename(config)),
        )

    return {
        "start_iteration": start_iteration,
        "end_iteration": end_iteration,
    }

## Create cropped video dataset version

In [ ]:
new_rows = []
rows = [row for _, row in df.iloc[:].iterrows()]

for row in tqdm(rows, total=len(rows)):
    new_rows.append(process_row(row))

# new_rows = Parallel(
#     n_jobs=8,           # use all cores
#     backend="loky",      # safe for IO + CPU
# )(
#     delayed(process_row)(row)
#     for row in tqdm(rows, total=len(rows))
# )

cropped_iter_df = pd.DataFrame(new_rows, index=df.index[:])
cropped_iter_df.head()

## Updated dataset metadata to cropped version

In [ ]:
cropped_df = pd.concat([df.iloc[:], cropped_iter_df], axis=1)

for i, row in tqdm(cropped_df.iterrows(), total=len(cropped_df)):
    rgb_path, display_path, label_path = row["rgb_path"], row["display_path"], row["annotations_path"]
    _accident_time, _accident_frame, _no_frames, _duration = row["accident_time"], row["accident_frame"], row["no_frames"], row["duration"]
    start_iteration, end_iteration = row["start_iteration"], row["end_iteration"]

    cropped_rgb_path = rgb_path.replace(INPUT_DIR, OUTPUT_DIR)
    cropped_display_path = display_path.replace(INPUT_DIR, OUTPUT_DIR)
    cropped_label_path = label_path.replace(INPUT_DIR, OUTPUT_DIR)

    assert osp.exists(cropped_rgb_path), f"{i}: {cropped_rgb_path} does not exist"
    assert osp.exists(cropped_display_path), f"{i}: {cropped_display_path} does not exist"
    assert osp.exists(cropped_label_path), f"{i}: {cropped_label_path} does not exist"

    annotation_file_data = load_json(cropped_label_path)
    base_data = annotation_file_data["base"]
    collision_data = annotation_file_data["collision"]
    assert base_data[0]["iteration"] == start_iteration, f"{i}: {base_data[0]['iteration']} != {start_iteration} iteration"

    accident_iteration, accident_bbox = collision_data[0]["iteration"], collision_data[0]["collision_bbox"]

    accident_frame = accident_iteration - start_iteration
    assert accident_frame == (_accident_frame + ITERATION_OFFSET - start_iteration), f"{i}: {accident_frame} != {_accident_frame} frame"
    accident_time = round(accident_frame / FPS, 4)

    fps, frame_count, duration = get_video_length(cropped_rgb_path)
    no_frames = end_iteration - start_iteration + 1  # start iteration inclusive
    assert no_frames == frame_count, f"{i}: {no_frames} != {frame_count} no_frames"
    # print("Duration diff:", _duration - duration)
    assert round(duration, 2) == round(_duration - (_no_frames - no_frames) / fps, 2), f"{i}: {duration} != {_duration} duration"

    cropped_df.loc[i, "rgb_path"] = cropped_rgb_path
    cropped_df.loc[i, "display_path"] = cropped_display_path
    cropped_df.loc[i, "annotations_path"] = cropped_label_path

    cropped_df.loc[i, "accident_time"] = accident_time
    cropped_df.loc[i, "accident_frame"] = accident_frame
    cropped_df.loc[i, "no_frames"] = no_frames
    cropped_df.loc[i, "duration"] = duration
    cropped_df.loc[i, "annotations_start_offset"] = base_data[0]["iteration"]


In [ ]:
cropped_df["accident_time"].min()

In [ ]:
cropped_df["accident_time"].hist(bins=100, figsize=(20, 10))

# Restructure the synthetic dataset to match the ACCIDENT dataset structure

In [ ]:
cropped_df.head()

In [ ]:
weather_mapping = {
    "ClearNoon": "clear",
    "ClearSunset": "sunset",
    "WetCloudySunset": "wet",
    "ClearNight": "night",
    "HardRainNoon": "rain",
}

cropped_df["weather"] = cropped_df["weather"].apply(lambda x: weather_mapping.get(x, x))

In [ ]:
SIM_DATASET_PATH = "../runs/sim_dataset"

rgb_video_dir = osp.join(SIM_DATASET_PATH, "videos")
annotations_dir = osp.join(SIM_DATASET_PATH, "video_annotations")

os.makedirs(rgb_video_dir, exist_ok=True)
os.makedirs(annotations_dir, exist_ok=True)

for i, row in tqdm(cropped_df.iterrows(), total=len(cropped_df)):
    rgb_path, label_path = row["rgb_path"], row["annotations_path"]
    scenario_map, location, weather, camera_index, accident_type = row["map"], row["location"], row["weather"], row["camera_index"], row["type"]

    target_rgb_file_name = f"{scenario_map}_{accident_type}_{weather}_{camera_index:02d}.mp4"
    target_label_file_name = f"{scenario_map}_{accident_type}_{weather}_{camera_index:02d}.json.gz"

    target_rgb_path = osp.join(rgb_video_dir, accident_type, target_rgb_file_name)
    os.makedirs(osp.dirname(target_rgb_path), exist_ok=True)
    target_label_path = osp.join(annotations_dir, target_label_file_name)

    annotation_file_data = load_json(label_path)
    base_data = annotation_file_data["base"]
    new_base_data = []
    for frame_info in base_data:  # Remove file_name
        new_base_data.append({
            "iteration": frame_info["iteration"],
            "objects": frame_info["objects"],
        })

    annotation_file_data["base"] = new_base_data

    if not osp.exists(target_rgb_path):
        shutil.copy(rgb_path, target_rgb_path)
    # shutil.copy(label_path, target_label_path)
    if not osp.exists(target_label_path):
        save_json(target_label_path, annotation_file_data, use_gzip=True)

    cropped_df.loc[i, "sim_rgb_path"] = target_rgb_path
    cropped_df.loc[i, "sim_annotations_path"] = target_label_path

cropped_df


In [ ]:
cropped_df_export = cropped_df.copy(deep=True)

In [ ]:
cropped_df_export["sim_rgb_path"].apply(lambda x: x.removeprefix(SIM_DATASET_PATH + "/"))

In [ ]:
cropped_df_export["scenario"] = cropped_df_export["scenario"].apply(lambda x: x.removesuffix("_video"))
cropped_df_export["rgb_path"] = cropped_df_export["sim_rgb_path"].apply(lambda x: x.removeprefix(SIM_DATASET_PATH + "/"))
# cropped_df_export["display_path"] = cropped_df_export["sim_display_path"].apply(lambda x: x.removeprefix(SIM_DATASET_PATH))
cropped_df_export["annotations_path"] = cropped_df_export["sim_annotations_path"].apply(lambda x: x.removeprefix(SIM_DATASET_PATH + "/"))

cropped_df_export["annotations_start_offset"] = cropped_df_export["start_iteration"]

cropped_df_export["camera_position"] = cropped_df_export["camera_index"]
cropped_df_export["timestamp"] = cropped_df_export["timestamp"].astype(str)

cols_to_drop = [
    "Unnamed: 0",
    "experiment",
    "sim_rgb_path",
    # "sim_display_path",
    "sim_annotations_path",
    "start_iteration",
    "end_iteration",
    "scenario",
    "camera_index",
    "timestamp",
    "location"
]

cropped_df_export.drop(cols_to_drop, axis=1, inplace=True)

cols_reorder = [
    # "timestamp",
    "rgb_path",
    # "display_path",
    "annotations_path", "type",
    "accident_time", "accident_frame", "center_x", "center_y", "x1", "y1", "x2", "y2",
    "map",
    # "location",
    "weather", "camera_position", "no_frames", "duration", "height", "width", "annotations_start_offset",
]
cropped_df_export = cropped_df_export[cols_reorder]

cropped_df_export.head()

In [ ]:
cropped_df_export.info()

In [ ]:
cropped_df_export.to_csv(osp.join(SIM_DATASET_PATH, "labels.csv"), index=False)